In [1]:
import os

BASE = r"D:\SIH26031\OnionModel2F\dataset2"
IMAGES_DIR = os.path.join(BASE, "images")
LABELS_CSV = os.path.join(BASE, "labels.csv")

In [10]:
import pandas as pd

df = pd.read_csv(LABELS_CSV)
image_files = set(os.listdir(IMAGES_DIR))
csv_files = set(df["filename"])

missing_images = csv_files - image_files
missing_labels = image_files - csv_files

print(f"Total rows in CSV: {len(df)}")
print(f"In CSV but image missing: {len(missing_images)}")
print(f"Image exists but not in CSV: {len(missing_labels)}")
if missing_images:
    print("Missing images:", list(missing_images)[:10])

Total rows in CSV: 1459
In CSV but image missing: 0
Image exists but not in CSV: 0


In [11]:
import os

missing_images = csv_files - image_files  # from your existing check

# get all files in the folder without caring about extension
all_files_no_ext = {}
for f in os.listdir(IMAGES_DIR):
    name, ext = os.path.splitext(f)
    all_files_no_ext[name] = ext

found_as_different_ext = []
truly_missing = []

for missing in missing_images:
    name, ext = os.path.splitext(missing)
    if name in all_files_no_ext:
        found_as_different_ext.append((missing, all_files_no_ext[name]))
    else:
        truly_missing.append(missing)

print(f"Found under a different extension: {len(found_as_different_ext)}")
print(f"Actually missing (not in folder at all): {len(truly_missing)}")
print("\nExamples of extension mismatches:")
for csv_name, actual_ext in found_as_different_ext[:10]:
    print(f"  CSV says: {csv_name}  →  Actual file: {os.path.splitext(csv_name)[0]}{actual_ext}")

Found under a different extension: 0
Actually missing (not in folder at all): 0

Examples of extension mismatches:


In [12]:
for csv_name, actual_ext in found_as_different_ext:
    old_path = os.path.join(IMAGES_DIR, os.path.splitext(csv_name)[0] + actual_ext)
    new_path = os.path.join(IMAGES_DIR, csv_name)  # renames to .jpg as CSV expects
    os.rename(old_path, new_path)

print(f"Renamed {len(found_as_different_ext)} files")

Renamed 0 files


In [13]:
from sklearn.model_selection import train_test_split

# 70% train, 20% validation, 10% test
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=(0.10/0.30), random_state=42)

train_df.to_csv(os.path.join(BASE, "train.csv"), index=False)
val_df.to_csv(os.path.join(BASE, "val.csv"), index=False)
test_df.to_csv(os.path.join(BASE, "test.csv"), index=False)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 1021 | Val: 291 | Test: 147


In [14]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms

DEFECT_COLUMNS = ["rotten", "sprouted", "cut_crack", "skin_damaged", "sunburned", "misshapen"]

class OnionDataset(Dataset):
    def __init__(self, csv_path, images_dir, transform=None):
        self.data = pd.read_csv(csv_path)
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(os.path.join(self.images_dir, row["filename"])).convert("RGB")
        if self.transform:
            image = self.transform(image)
        labels = torch.tensor(row[DEFECT_COLUMNS].values.astype("float32"))
        return image, labels

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = OnionDataset(os.path.join(BASE, "train.csv"), IMAGES_DIR, transform=train_transform)
val_dataset   = OnionDataset(os.path.join(BASE, "val.csv"), IMAGES_DIR, transform=eval_transform)
test_dataset  = OnionDataset(os.path.join(BASE, "test.csv"), IMAGES_DIR, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

In [15]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(DEFECT_COLUMNS))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()  # correct loss for multi-label
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [16]:
from sklearn.metrics import precision_recall_fscore_support
import numpy as np

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.numpy())
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average=None, zero_division=0
    )
    for i, defect in enumerate(DEFECT_COLUMNS):
        print(f"  {defect:15s} Precision: {precision[i]:.2f}  Recall: {recall[i]:.2f}  F1: {f1[i]:.2f}")
    return all_preds, all_labels

num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"\nEpoch {epoch+1}/{num_epochs} | Train Loss: {running_loss/len(train_loader):.4f}")
    print("Validation metrics:")
    evaluate(model, val_loader)


Epoch 1/20 | Train Loss: 0.5445
Validation metrics:
  rotten          Precision: 0.74  Recall: 0.24  F1: 0.36
  sprouted        Precision: 1.00  Recall: 0.96  F1: 0.98
  cut_crack       Precision: 0.87  Recall: 0.67  F1: 0.76
  skin_damaged    Precision: 0.73  Recall: 0.67  F1: 0.70
  sunburned       Precision: 0.69  Recall: 0.34  F1: 0.46
  misshapen       Precision: 0.50  Recall: 0.03  F1: 0.06

Epoch 2/20 | Train Loss: 0.3489
Validation metrics:
  rotten          Precision: 0.68  Recall: 0.40  F1: 0.50
  sprouted        Precision: 0.98  Recall: 0.98  F1: 0.98
  cut_crack       Precision: 0.83  Recall: 0.68  F1: 0.75
  skin_damaged    Precision: 0.82  Recall: 0.72  F1: 0.76
  sunburned       Precision: 0.61  Recall: 0.48  F1: 0.54
  misshapen       Precision: 1.00  Recall: 0.27  F1: 0.43

Epoch 3/20 | Train Loss: 0.2867
Validation metrics:
  rotten          Precision: 0.66  Recall: 0.53  F1: 0.59
  sprouted        Precision: 1.00  Recall: 0.98  F1: 0.99
  cut_crack       Precision: 

In [17]:
best_f1 = 0
best_epoch = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"\nEpoch {epoch+1}/{num_epochs} | Train Loss: {running_loss/len(train_loader):.4f}")
    print("Validation metrics:")
    all_preds, all_labels = evaluate(model, val_loader)

    # average F1 across all defects as the checkpoint criterion
    from sklearn.metrics import f1_score
    avg_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    if avg_f1 > best_f1:
        best_f1 = avg_f1
        best_epoch = epoch + 1
        torch.save(model.state_dict(), r"D:\SIH26031\OnionModel2F\onion_defect_classifier_BEST.pt")
        print(f"  → New best model saved (avg F1: {avg_f1:.3f})")

print(f"\nBest model was epoch {best_epoch} with avg F1 {best_f1:.3f}")


Epoch 1/20 | Train Loss: 0.0512
Validation metrics:
  rotten          Precision: 0.83  Recall: 0.83  F1: 0.83
  sprouted        Precision: 1.00  Recall: 0.98  F1: 0.99
  cut_crack       Precision: 0.95  Recall: 0.79  F1: 0.86
  skin_damaged    Precision: 0.85  Recall: 0.78  F1: 0.82
  sunburned       Precision: 0.64  Recall: 0.74  F1: 0.69
  misshapen       Precision: 0.73  Recall: 0.67  F1: 0.70
  → New best model saved (avg F1: 0.814)

Epoch 2/20 | Train Loss: 0.0511
Validation metrics:
  rotten          Precision: 0.79  Recall: 0.84  F1: 0.82
  sprouted        Precision: 1.00  Recall: 0.98  F1: 0.99
  cut_crack       Precision: 0.97  Recall: 0.81  F1: 0.88
  skin_damaged    Precision: 0.90  Recall: 0.77  F1: 0.83
  sunburned       Precision: 0.68  Recall: 0.64  F1: 0.66
  misshapen       Precision: 0.68  Recall: 0.64  F1: 0.66

Epoch 3/20 | Train Loss: 0.0483
Validation metrics:
  rotten          Precision: 0.81  Recall: 0.86  F1: 0.83
  sprouted        Precision: 1.00  Recall: 0.9

In [18]:
model.load_state_dict(torch.load(r"D:\SIH26031\OnionModel2F\onion_defect_classifier_BEST.pt"))
print("Final test set performance (best checkpoint):")
evaluate(model, test_loader)

Final test set performance (best checkpoint):
  rotten          Precision: 0.83  Recall: 0.83  F1: 0.83
  sprouted        Precision: 1.00  Recall: 0.96  F1: 0.98
  cut_crack       Precision: 0.83  Recall: 0.83  F1: 0.83
  skin_damaged    Precision: 0.87  Recall: 0.85  F1: 0.86
  sunburned       Precision: 0.56  Recall: 0.69  F1: 0.62
  misshapen       Precision: 0.71  Recall: 0.63  F1: 0.67


(array([[1., 0., 1., 1., 0., 0.],
        [0., 0., 1., 1., 0., 0.],
        [0., 0., 1., 1., 1., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 1., 1., 0.],
        [0., 0., 0., 1., 1., 0.],
        [0., 0., 0., 1., 1., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 1., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 1., 0., 1., 1., 0.],
        [1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 1., 0.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 1., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0.],
        [0., 0

In [19]:
print(df["sunburned"].value_counts())

sunburned
0    1113
1     346
Name: count, dtype: int64
